# Nyaya — Eval-v1 + BhashaBench comparison run

Answers the question none of the published numbers could: **did the fine-tune help?**

Eval-v0's metric scored its own gold answers at ~10.7%, so base/v3/v4 were pinned
within a 2-answer spread by the ruler, not by the models. Eval-v1 has a 100% gold
ceiling, so a real difference can now show up.

**Before running:** Settings → Accelerator → **GPU T4 x2**, Internet → **On**,
and add your HF read token as a Kaggle Secret named `HF_TOKEN`.

Runtime is roughly 60–90 min on one T4. Every run saves `predictions.jsonl`, so
scoring can be redone later on CPU without re-running any model.

In [ ]:
# --- setup -------------------------------------------------------------
!pip -q install -U transformers accelerate peft sentence-transformers rank_bm25 huggingface_hub

import os, subprocess, sys

# Long, highly variable RAG prompts fragment the allocator on a 14.5 GiB T4.
# Set before any torch import; inherited by the subprocesses we spawn.
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"  # older torch

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle Secrets")
except Exception as exc:
    print("No HF_TOKEN secret found — gated datasets (BhashaBench) will skip:", exc)

REPO = "https://github.com/JitendraJha98/nyaya-model.git"
if not os.path.exists("/kaggle/working/nyaya-model"):
    subprocess.run(["git", "clone", "--depth", "1", REPO,
                    "/kaggle/working/nyaya-model"], check=True)
os.chdir("/kaggle/working/nyaya-model")
sys.path.insert(0, "src")
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

In [ ]:
# --- build Eval-v1 + GPU preflight -------------------------------------
# Only the PUBLIC half is committed to GitHub; the private half is gitignored.
# Regenerating from v0 here reproduces both halves deterministically.
import subprocess, sys

import torch


def run(cmd):
    """Run a pipeline step and FAIL LOUDLY.

    `!python ...` in Jupyter swallows non-zero exits, so a crashed eval run
    looked like success and only surfaced three cells later as a confusing
    FileNotFoundError on the results file. Surface it where it happens.
    """
    print("$", " ".join(cmd), flush=True)
    proc = subprocess.run(cmd, text=True)
    if proc.returncode != 0:
        raise RuntimeError(f"step failed (exit {proc.returncode}): {' '.join(cmd)}")


run([sys.executable, "scripts/25_build_eval_v1.py"])

# Preflight: Kaggle can hand out a P100 (sm_60), and the preinstalled torch is
# built for sm_70+. Every model call then dies with "no kernel image is
# available for execution on the device". Catch it here, not 40 minutes in.
if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Settings -> Accelerator -> GPU T4 x2.")

name = torch.cuda.get_device_name(0)
major, minor = torch.cuda.get_device_capability(0)
supported = torch.cuda.get_arch_list()
print(f"GPU: {name}  (sm_{major}{minor})")
print(f"torch {torch.__version__} supports: {supported}")

if f"sm_{major}{minor}" not in supported:
    raise RuntimeError(
        f"{name} is sm_{major}{minor}, which this torch build does not support "
        f"({supported}). Switch the accelerator to GPU T4 x2 and re-run.")
print("preflight OK")

## 1. Eval-v1: base vs the published release

Same retriever, same prompt, same questions — the only variable is the weights.
That is what makes the comparison meaningful.

In [ ]:
# --- SMOKE FIRST: 8 questions, timed, extrapolated ----------------------
# Runs 1-4 all failed or crawled for environment reasons a local test cannot
# catch (GPU model, memory ceiling, emulated dtypes). This cell spends ~3-5
# minutes proving the REAL environment end to end -- model download, retrieval,
# generation, scoring -- and aborts with a projection if the full run would
# blow the session budget. Never skip it: 5 minutes here beats 4.5 hours lost.
import time

N_SMOKE = 8
FULL_N = 413            # gradeable Eval-v1 records
N_MODELS = 2            # base + v3
BUDGET_S = 6 * 3600     # abort if projected beyond this

t0 = time.time()
run([sys.executable, "scripts/26_eval_v1_run.py", "--adapter", "none",
     "--dense", "--k", "8", "--limit", str(N_SMOKE), "--batch-size", "4",
     "--label", "smoke"])
smoke_s = time.time() - t0

per_q = smoke_s / N_SMOKE
projected = per_q * FULL_N * N_MODELS
print(f"\nsmoke: {smoke_s:.0f}s for {N_SMOKE} questions "
      f"({per_q:.1f}s/question, includes model download)")
print(f"projected full run (~{N_MODELS} models x {FULL_N} q): "
      f"~{projected/3600:.1f} h  (budget {BUDGET_S/3600:.0f} h)")

if projected > BUDGET_S:
    raise RuntimeError(
        f"projected {projected/3600:.1f}h exceeds the {BUDGET_S/3600:.0f}h budget "
        f"-- something is running far slower than it should (emulated dtype? "
        f"wrong GPU?). Fix that before burning the session.")
print("smoke OK -- proceeding to full runs")

In [ ]:
# Base model + dense RAG
#
# batch 4 with the OOM auto-halving retry as the safety net. A T4 has 14.5 GiB;
# the 3B takes ~6.2 and the e5 retriever ~1, so batch 8 OOMed on long k=8
# contexts. The retry means a too-large batch costs a retry, not the run.
run([sys.executable, "scripts/26_eval_v1_run.py", "--adapter", "none",
     "--dense", "--k", "8", "--split", "all", "--batch-size", "4",
     "--label", "base"])

In [ ]:
# The published merged release, pulled straight from the Hub
run([sys.executable, "scripts/26_eval_v1_run.py",
     "--model", "NyayaLabs98/nyaya-3b-v3", "--adapter", "none",
     "--dense", "--k", "8", "--split", "all", "--batch-size", "4",
     "--label", "nyaya-3b-v3"])

In [ ]:
# Optional size-matched general baseline — DISABLED by default.
#
# It runs a third full pass and roughly doubles wall-clock. Enable it only once
# base vs v3 has landed; that comparison is the point of this notebook and must
# not be put at risk of hitting Kaggle's session limit.
RUN_OPTIONAL_BASELINE = False

if RUN_OPTIONAL_BASELINE:
    # Phi-3.5-mini is MIT-licensed and NOT gated. Llama-3.2-3B is gated and
    # needs Meta's licence accepted on the same account as HF_TOKEN.
    proc = subprocess.run(
        [sys.executable, "scripts/26_eval_v1_run.py",
         "--model", "microsoft/Phi-3.5-mini-instruct", "--adapter", "none",
         "--dense", "--k", "8", "--split", "all", "--batch-size", "4",
         "--label", "phi-3.5-mini"], text=True)
    if proc.returncode != 0:
        print(f"\n!! optional baseline skipped (exit {proc.returncode}) — continuing")
else:
    print("optional baseline disabled — set RUN_OPTIONAL_BASELINE = True to include it")

## 2. BhashaBench-Legal — external, MCQ-scored

MCQ scoring is exact, so there is no phrase-matching problem and the numbers are
directly comparable against any other model. This is the benchmark to quote publicly.

In [ ]:
import json, re, torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# Native bf16 only on Ampere+ (sm_80). NOT torch.cuda.is_bf16_supported(),
# which defaults to including_emulation=True and returns True on a T4, where
# bf16 is emulated in software and several times slower.
DTYPE = (torch.bfloat16
         if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
         else torch.float16)
print("dtype:", DTYPE)

# Schema verified against the actual dataset: columns are
# option_a..option_d / correct_answer, and it ships as separate
# English and Hindi configs -- there is no single "test" split.
SUBSET = 1000          # per language; raise once the pipeline is proven
LANGUAGES = ["English", "Hindi"]
LETTERS = ("A", "B", "C", "D")


def mcq_prompt(row):
    opts = "\n".join(
        f"{letter}. {row['option_' + letter.lower()]}"
        for letter in LETTERS
        if row.get("option_" + letter.lower()) not in (None, ""))
    return (f"{row['question']}\n{opts}\n\n"
            "Answer with the single letter of the correct option.")


def run_mcq(bench, model_id, label, batch_size=16):
    tok = AutoTokenizer.from_pretrained(model_id, padding_side="left")
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        model_id, dtype=DTYPE, device_map="auto").eval()

    correct = total = unparsed = 0
    rows = []
    for start in range(0, len(bench), batch_size):
        batch = bench.select(range(start, min(start + batch_size, len(bench))))
        texts = [tok.apply_chat_template(
            [{"role": "user", "content": mcq_prompt(r)}],
            tokenize=False, add_generation_prompt=True) for r in batch]
        enc = tok(texts, return_tensors="pt", padding=True).to(model.device)
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=8, do_sample=False,
                                 pad_token_id=tok.pad_token_id)
        decoded = tok.batch_decode(out[:, enc["input_ids"].shape[1]:],
                                   skip_special_tokens=True)
        for row, text in zip(batch, decoded):
            match = re.search(r"\b([ABCD])\b", text.upper())
            pred = match.group(1) if match else None
            unparsed += int(pred is None)
            gold = str(row["correct_answer"]).strip().upper()[:1]
            correct += int(pred == gold)
            total += 1
            rows.append({"pred": pred, "gold": gold, "raw": text,
                         "language": row.get("language"),
                         "domain": row.get("subject_domain")})
        print(f"\r{label}: {total}/{len(bench)} acc={correct/max(1,total):.1%}",
              end="", flush=True)

    del model
    torch.cuda.empty_cache()
    with open(f"/kaggle/working/bhashabench_{label}.jsonl", "w") as fh:
        for r in rows:
            fh.write(json.dumps(r, ensure_ascii=False) + "\n")

    # A high unparsed count means the model ignored the format, not that it is
    # ignorant -- report it rather than silently scoring those as wrong.
    print(f"\n{label}: {correct}/{total} = {correct/total:.1%}"
          f"   (unparseable replies: {unparsed})")

    by_lang = {}
    for lang in {r["language"] for r in rows}:
        subset = [r for r in rows if r["language"] == lang]
        by_lang[lang] = sum(r["pred"] == r["gold"] for r in subset) / len(subset)
    print("   by language:", {k: f"{v:.1%}" for k, v in by_lang.items()})
    return {"accuracy": correct / total, "unparsed": unparsed, "by_language": by_lang}


# BhashaBench is gated and needs HF_TOKEN. It must NEVER take down the run:
# the Eval-v1 results above are the primary output and are already on disk.
scores = {}
try:
    from datasets import concatenate_datasets, load_dataset

    parts = []
    for config in LANGUAGES:
        ds = load_dataset("bharatgenai/BhashaBench-Legal", config, split="test")
        if SUBSET and len(ds) > SUBSET:
            ds = ds.shuffle(seed=0).select(range(SUBSET))
        parts.append(ds)
        print(f"{config}: {len(ds)} questions")
    bench = concatenate_datasets(parts)
    print(len(bench), "total |", bench.column_names)

    for model_id, label in [("Qwen/Qwen2.5-3B-Instruct", "base"),
                            ("NyayaLabs98/nyaya-3b-v3", "nyaya-3b-v3")]:
        scores[label] = run_mcq(bench, model_id, label)

    with open("/kaggle/working/bhashabench_scores.json", "w") as fh:
        json.dump(scores, fh, indent=2)
    print(json.dumps(scores, indent=2))

except Exception as exc:
    print(f"\n!! BhashaBench SKIPPED: {type(exc).__name__}: {exc}")
    print("!! Most likely cause: HF_TOKEN secret missing, or the dataset terms")
    print("!! have not been accepted on huggingface.co with that account.")
    print("!! The Eval-v1 results are unaffected — continue to the results cell.")

## 3. Results — download these

`predictions.jsonl` files matter most: they let scoring be revised later without a GPU.
That is exactly what was missing when the v0 scorer turned out to be broken.

In [ ]:
import glob, json, pathlib, shutil

out = pathlib.Path("/kaggle/working/nyaya-eval-v1-results")
out.mkdir(exist_ok=True)

results = json.load(open("reports/eval_v1_results.json"))
print(f"{'run':<18} {'fact_recall':>12} {'citation':>10} {'all_facts':>10} {'forbidden':>10}")
for label, payload in results.items():
    m = payload["metrics"]
    print(f"{label:<18} {m['fact_recall']:>11.1%} {m['citation_accuracy']:>10.1%} "
          f"{m['all_facts_accuracy']:>10.1%} {m['forbidden_violation_rate']:>10.1%}")

# Headline comparison: same retriever, same prompt, only the weights differ.
if "base" in results and "nyaya-3b-v3" in results:
    b = results["base"]["metrics"]["fact_recall"]
    v = results["nyaya-3b-v3"]["metrics"]["fact_recall"]
    n = results["base"]["metrics"]["scored_total"]
    print(f"\nfact_recall  base {b:.1%}  ->  v3 {v:.1%}   delta {v - b:+.1%}  (n={n})")
    print("Treat a delta under ~2 points at this n as noise, not a win.")

shutil.copy("reports/eval_v1_results.json", out)
for run in pathlib.Path("outputs/eval-v1").glob("*/predictions.jsonl"):
    shutil.copy(run, out / f"{run.parent.name}_predictions.jsonl")
for extra in glob.glob("/kaggle/working/bhashabench_*"):
    shutil.copy(extra, out)
shutil.copy("reports/eval_v1_curation.json", out)

shutil.make_archive("/kaggle/working/nyaya-eval-v1-results", "zip", out)
print("\nFiles collected:", sorted(p.name for p in out.iterdir()))
print("Download /kaggle/working/nyaya-eval-v1-results.zip from the Output tab.")